# Feature Extraction

Data representation plays a critical role in the performance of many machine learning methods in machine learning. The data representation of network traffic often determines the effectiveness of these models as much as the model itself. The wide range of novel events that network operators need to detect (e.g., attacks, malware, new applications, changes in traffic demands) introduces the possibility for a broad range of possible models and data representations.

[NetML](https://pypi.org/project/netml/) is an open-source tool and end-to-end pipeline for anomaly detection in network traffic. This notebook walks through the use of that library.

First, let us load the library.

In [3]:
!conda activate netml-311
!conda install -c conda-forge numpy scapy netaddr pandas -y
!pip install --no-build-isolation netml==0.7.1



CondaError: Run 'conda init' before 'conda activate'

2 channel Terms of Service accepted
Channels:
 - conda-forge
 - defaults
Platform: osx-arm64
Solving environment: done

# All requested packages already installed.

  Using cached netml-0.7.1-py3-none-any.whl.metadata (11 kB)
  Using cached numpy-2.0.2.tar.gz (18.9 MB)
  Preparing metadata (pyproject.toml) ... done
ERROR: Exception:
Traceback (most recent call last):
  File "/Users/chrislowzx/miniconda3/lib/python3.13/site-packages/pip/_internal/cli/base_command.py", line 107, in _run_wrapper
    status = _inner_run()
  File "/Users/chrislowzx/miniconda3/lib/python3.13/site-packages/pip/_internal/cli/base_command.py", line 98, in _inner_run
    return self.run(options, args)
           ~~~~~~~~^^^^^^^^^^^^^^^
  File "/Users/chrislowzx/miniconda3/lib/python3.13/site-packages/pip/_internal/cli/req_command.py", line 71, in wrapper
    return func(self, options, args)
  File "/Users/chrislowzx/miniconda3/lib/python3.13/site-packages/pi

In [9]:
!python -m ipykernel install --user --name netml-311 --display-name "Python (netml-311)"


Installed kernelspec netml-311 in /Users/chrislowzx/Library/Jupyter/kernels/netml-311


In [2]:
!conda init
!conda activate netml-311
!which python
!python --version


no change     /Users/chrislowzx/miniconda3/condabin/conda
no change     /Users/chrislowzx/miniconda3/bin/conda
no change     /Users/chrislowzx/miniconda3/bin/conda-env
no change     /Users/chrislowzx/miniconda3/bin/activate
no change     /Users/chrislowzx/miniconda3/bin/deactivate
no change     /Users/chrislowzx/miniconda3/etc/profile.d/conda.sh
no change     /Users/chrislowzx/miniconda3/etc/fish/conf.d/conda.fish
no change     /Users/chrislowzx/miniconda3/shell/condabin/Conda.psm1
no change     /Users/chrislowzx/miniconda3/shell/condabin/conda-hook.ps1
no change     /Users/chrislowzx/miniconda3/lib/python3.13/site-packages/xontrib/conda.xsh
no change     /Users/chrislowzx/miniconda3/etc/profile.d/conda.csh
no change     /Users/chrislowzx/.bash_profile
No action taken.

CondaError: Run 'conda init' before 'conda activate'

/Users/chrislowzx/miniconda3/bin/python
Python 3.13.5


In [2]:
!pip install netml

  Using cached netml-0.7.1-py3-none-any.whl.metadata (11 kB)
  Using cached numpy-2.0.2.tar.gz (18.9 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... error
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [635 lines of output]
      + /Users/chrislowzx/miniconda3/bin/python3.13 /private/var/folders/vp/npcvxcc52xqfjgmswcdds6yc0000gn/T/pip-install-_ktsdecf/numpy_31c5d85ff9fa4eb9b628332ee4b1008b/vendored-meson/meson/meson.py setup /private/var/folders/vp/npcvxcc52xqfjgmswcdds6yc0000gn/T/pip-install-_ktsdecf/numpy_31c5d85ff9fa4eb9b628332ee4b1008b /private/var/folders/vp/npcvxcc52xqfjgmswcdds6yc0000gn/T/pip-install-_ktsdecf/numpy_31c5d85ff9fa4eb9b628332ee4b1008b/.mesonpy-pehn88kj -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=/private/var/folders/vp/npcvxcc52xqfjg

In [1]:
import logging
logging.getLogger("scapy.runtime").setLevel(logging.ERROR)

from netml.pparser.parser import PCAP
from netml.utils.tool import dump_data, load_data

import pandas as pd

/Users/chrislowzx/miniconda3/envs/netml-311/lib/python3.11/site-packages/scapy/layers/ipsec.py:512: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  cipher=algorithms.TripleDES,
/Users/chrislowzx/miniconda3/envs/netml-311/lib/python3.11/site-packages/scapy/layers/ipsec.py:516: CryptographyDeprecationWarning: TripleDES has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.TripleDES and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  cipher=algorithms.TripleDES,


## Specify a Packet Capture File

Create a pcap data structure for which we would like to extract features. You could do this based on the packet capture files that we have been using in previous hands assignments. Any packet capture file will suffice, however.

You can define the minumum number of packets that you want to include in each flow.

In [ ]:
pcap_path = "data/log4j.pcap"        
min_pkts  = 2                   

pcap = PCAP(pcap_path, flow_ptks_thres=min_pkts)
pcap.pcap2flows()
print(f"Flows built: {len(pcap.flows)}")

Flows built: 4795


## Convert the Packet Capture Into Flows

Find the function in `netml` that converts the pcap file into flows. Examing the resulting data structure. What does it contain?

In [10]:
type(pcap.flows)              

list

In [8]:
pcap.flows[:3]                # preview the first few flow objects

[(('128.14.134.170', '198.71.247.91', 57468, 80, 6),
  [<Ether  dst=00:16:3c:f1:fd:6d src=64:9e:f3:be:db:66 type=IPv4 |<IP  version=4 ihl=5 tos=0x0 len=60 id=35444 flags= frag=0 ttl=52 proto=tcp chksum=0x37ec src=128.14.134.170 dst=198.71.247.91 |<TCP  sport=57468 dport=http seq=3705618145 ack=0 dataofs=10 reserved=0 flags=S window=29200 chksum=0x13ad urgptr=0 options=[('MSS', 1460), ('SAckOK', b''), ('Timestamp', (1287894165, 0)), ('NOP', None), ('WScale', 7)] |>>>,
   <Ether  dst=00:16:3c:f1:fd:6d src=64:9e:f3:be:db:66 type=IPv4 |<IP  version=4 ihl=5 tos=0x0 len=52 id=35445 flags= frag=0 ttl=52 proto=tcp chksum=0x37f3 src=128.14.134.170 dst=198.71.247.91 |<TCP  sport=57468 dport=http seq=3705618146 ack=3613341764 dataofs=8 reserved=0 flags=A window=229 chksum=0x9891 urgptr=0 options=[('NOP', None), ('NOP', None), ('Timestamp', (1287894168, 2736285763))] |>>>,
   <Ether  dst=00:16:3c:f1:fd:6d src=64:9e:f3:be:db:66 type=IPv4 |<IP  version=4 ihl=5 tos=0x0 len=257 id=35446 flags= frag=0 

## Explore the Flows

How many flows are in your data structure?

In [11]:
print(f"Flows built: {len(pcap.flows)}")

Flows built: 4795


In [12]:
pcap.pcap2pandas()
df = pcap.df
df.head()


,datetime,dns_query,dns_resp,ip_dst,ip_dst_int,ip_src,ip_src_int,is_dns,length,mac_dst,mac_dst_int,mac_src,mac_src_int,port_dst,port_src,protocol,time,time_normed
0,2021-12-15 15:35:00,None,None,198.71.247.91,3326605147,128.14.134.170,2148435626,False,74,00:16:3c:f1:fd:6d,95511772525,64:9e:f3:be:db:66,110633856981862,80.0,57468.0,TCP,1639604100.237882,0.000000
1,2021-12-15 15:35:00,None,None,128.14.134.170,2148435626,198.71.247.91,3326605147,False,74,64:9e:f3:be:db:66,110633856981862,00:16:3c:f1:fd:6d,95511772525,57468.0,80.0,TCP,1639604100.237939,0.000057
2,2021-12-15 15:35:00,None,None,198.71.247.91,3326605147,128.14.134.170,2148435626,False,66,00:16:3c:f1:fd:6d,95511772525,64:9e:f3:be:db:66,110633856981862,80.0,57468.0,TCP,1639604100.249425,0.011543
3,2021-12-15 15:35:00,None,None,198.71.247.91,3326605147,128.14.134.170,2148435626,False,271,00:16:3c:f1:fd:6d,95511772525,64:9e:f3:be:db:66,110633856981862,80.0,57468.0,TCP,1639604100.249475,0.011593
4,2021-12-15 15:35:00,None,None,128.14.134.170,2148435626,198.71.247.91,3326605147,False,66,64:9e:f3:be:db:66,110633856981862,00:16:3c:f1:fd:6d,95511772525,57468.0,80.0,TCP,1639604100.249525,0.011643


What other information does the flow data structure contain, for each flow?

## Extract Features from Each Flow

Use the `netml` library to extract features from each flow. 

The [documentation](https://pypi.org/project/netml/) and [accompanying paper](https://arxiv.org/pdf/2006.16993.pdf) provide examples of features that you can try to extract. 

First try to extract the inter-arrival times for each flow.

### Interarrival Times

In [16]:
import logging
logging.getLogger("scapy.runtime").setLevel(logging.ERROR)

from netml.pparser.parser import PCAP
from netml.utils.tool import dump_data
import pandas as pd

pcap_path = "data/log4j.pcap"    # change if needed
min_pkts  = 2               # minimum packets per flow

pcap = PCAP(pcap_path, flow_ptks_thres=min_pkts)
pcap.pcap2flows()           # convert packets to flows
print(f"Flows: {len(pcap.flows)}")

# Inter-arrival time features per flow
pcap.flow2features('IAT', fft=False, header=False)
iat = pcap.features.copy()
print("IAT feature shape:", iat.shape)

print(iat.columns.tolist())
iat.describe().T.head(20)



Flows: 4795
IAT feature shape: (4795, 5)


AttributeError: 'numpy.ndarray' object has no attribute 'columns'

### Explore the Per-Flow Features

Inspect and print the features for each flow. (If you feel compelled: Get fancy! Plot distributions, etc. Whatever you like!)

### Other Features and Options

1. Try some of the other features in the `netml` library.

  Here are some of the other possibilities, which can be passed to the library:
  * IAT: A flow is represented as a timeseries of inter-arrival times between packets, i.e., elapsed time in seconds between any two packets in the flow.   
  *  STATS: A flow is represented as a set of statistical quantities. We choose ten of the most common such
statistics in the literature: flow duration, number of packets sent per second, number of bytes
per second, and various statistics on packet sizes within each flow: mean, standard deviation, inter-quartile range,
minimum, and maximum.
  * SIZE: A flow is represented as a timeseries of packet sizes in bytes, with one sample per packet. 
  * SAMP-NUM: A flow is partitioned into small intervals of equal length 𝛿𝑡, and the number of packets in each interval is recorded; thus, a flow is represented as a timeseries of packet counts in small time intervals, with one sample per time interval. Here, 𝛿𝑡 might be viewed as a choice of sampling rate for the timeseries, hence the nomenclature.
  * SAMP-SIZE: A flow is partitioned into time intervals of equal length 𝛿𝑡, and the total packet size (i.e., byte count) in each interval is recorded; thus, a flow is represented as a timeseries of byte counts in small time intervals, with one sample per time interval.
  

  
2. One of the challenges with providing packet traces to models involve ensuring that all feature vectors are of the same length. The `netml` libary will do that for you, but there are a number of different ways to solve the problem. What do some of the following options do?  Explore how different settings of the following affect the dimensionality of your resulting feature vector.

 * flow_ptks_thres
 * q_interval

## Thought Questions

What other features might you want to extract from packet captures that are not provided by the `netml` library?